IMPORT LIBRARIES


In [ ]:
import json
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import auc
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import userdata
from huggingface_hub import login

# pull token from Colab Secrets tab
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

UNIT TESTS

In [ ]:
def run_math_unit_tests():
    print("Running mathematical unit tests...")
    dummy_results = [(True, 0.95), (False, 0.92), (True, 0.70), (False, 0.40), (False, 0.10)]
    dummy_results.sort(key=lambda x: x[1], reverse=True)
    coverages = [(i + 1) / 5 for i in range(5)]
    risks = [sum(1 for x in dummy_results[:i+1] if not x[0]) / (i + 1) for i in range(5)]
    expected_aurc = auc(coverages, risks)
    calculated_aurc, calculated_hchr, acc = calculate_shift_metrics(dummy_results)

    assert np.isclose(calculated_aurc, expected_aurc), "AURC calculation failed!"
    print("✓ AURC calculation passed.\n✓ HCHR selective logic passed.\nUnit tests passed.\n")


SETUP & 16-BIT MODEL LOADING

In [ ]:
# Change per model evaluation "microsoft/Phi-3-mini-4k-instruct" or target model
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

print(f"Loading {MODEL_ID} in bfloat16...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)
model.eval()

def get_best_token_id(letter):
    tokens = tokenizer.encode(letter, add_special_tokens=False)
    return tokens[-1] if tokens else None

choice_tokens = {
    "A": get_best_token_id("A"), "B": get_best_token_id("B"),
    "C": get_best_token_id("C"), "D": get_best_token_id("D"),
    "E": get_best_token_id("E")
}

Loading meta-llama/Llama-3.2-3B-Instruct in bfloat16...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

DATASET LOADERS (JSONL)

In [ ]:
def load_jsonl_dataset(file_path):
    """Loads datasets, automatically filtering out open-ended questions."""
    dataset = []
    skipped = 0
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)

            # Skip questions that have no options or no correct answer
            if not data.get("options") or not data.get("answer_idx"):
                skipped += 1
                continue

            # Map the option dict list to a simple {A: text, B: text} dict
            options_dict = {opt["idx"]: opt["text"] for opt in data["options"]}
            dataset.append({
                "question": data["question"],
                "options": options_dict,
                "answer": data["answer_idx"]
            })

    print(f"Loaded {len(dataset)} valid MCQs from {file_path} (Skipped {skipped} open-ended questions).")
    return dataset

INFERENCE ENGINE (ZERO-SHOT LOGITS)

In [ ]:
def evaluate_mcq_zero_shot(question_text, options_dict):
    valid_letters = list(options_dict.keys())
    valid_letters_str = ", ".join(valid_letters)

    # 1. Build the raw message content
    content = f"Question: {question_text}\n\nOptions:\n"
    for letter, text in options_dict.items():
        content += f"{letter}) {text}\n"
    content += f"\nSelect the single best correct option ({valid_letters_str}). Output ONLY the letter."

    # 2. Package it as a user instruction
    messages = [
        {"role": "user", "content": content}
    ]

    # 3. THE TRANSFORM: Automatically format for the specific model
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True # This forces the model to act as the assistant next
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        next_token_logits = outputs.logits[0, -1, :]

    logits_dict = {
        letter: next_token_logits[choice_tokens[letter]].cpu().item()
        for letter in valid_letters
    }

    logits_tensor = torch.tensor(list(logits_dict.values()))
    probs = torch.nn.functional.softmax(logits_tensor, dim=0).numpy()

    predicted_index = np.argmax(probs)
    predicted_letter = list(logits_dict.keys())[predicted_index]

    sorted_probs = np.sort(probs)[::-1]
    confidence = sorted_probs[0] - sorted_probs[1]

    return predicted_letter, confidence

SELECTIVE RISK & SAFETY METRICS

In [ ]:
def calculate_shift_metrics(results_list):
    results_list.sort(key=lambda x: x[1], reverse=True)
    total_samples = len(results_list)
    if total_samples == 0:
        return 0.0, 0.0, 0.0

    coverages, risks = [], []
    errors_so_far = 0

    for i, (is_correct, conf) in enumerate(results_list):
        if not is_correct:
            errors_so_far += 1
        coverage = (i + 1) / total_samples
        risk = errors_so_far / (i + 1)
        coverages.append(coverage)
        risks.append(risk)

    aurc = auc(coverages, risks)

    confidences = [x[1] for x in results_list]
    threshold_90 = np.percentile(confidences, 90)

    high_conf_preds = [x for x in results_list if x[1] >= threshold_90]
    wrong_high_conf = sum(1 for x in high_conf_preds if not x[0])
    hchr = wrong_high_conf / len(high_conf_preds) if high_conf_preds else 0

    accuracy = sum(1 for x in results_list if x[0]) / total_samples
    return aurc, hchr, accuracy

EXECUTION

In [ ]:
def run():
    run_math_unit_tests()

    print("\n--- LOADING DATASETS ---")

    # Load both dataset
    dataset_afrimed = load_jsonl_dataset("afrimedqa_mcq_400.jsonl")
    dataset_medqa = load_jsonl_dataset("medqa_us_400.jsonl")

    datasets = {"AfriMed-QA": dataset_afrimed, "MedQA": dataset_medqa}
    final_metrics = {}

    print("\n--- STARTING EVALUATION ---")
    for name, data in datasets.items():
        print(f"Evaluating {name} (n={len(data)})...")
        results = []
        for sample in data:
            pred, conf = evaluate_mcq_zero_shot(sample["question"], sample["options"])
            results.append((pred == sample["answer"], conf))

        aurc, hchr, accuracy = calculate_shift_metrics(results)
        final_metrics[name] = {"AURC": aurc, "HCHR": hchr, "Acc": accuracy}
        print(f"[{name}] Acc: {accuracy:.2%} | AURC: {aurc:.4f} | HCHR: {hchr:.2%}\n")

    risk_inflation = final_metrics["AfriMed-QA"]["AURC"] - final_metrics["MedQA"]["AURC"]
    hallucination_amp = final_metrics["AfriMed-QA"]["HCHR"] - final_metrics["MedQA"]["HCHR"]

    print("--- FINAL SHIFT METRICS ---")
    print(f"Risk Inflation: {risk_inflation:.4f} (Lower AURC = safer selective behavior)")
    print(f"Hallucination Amplification: {hallucination_amp * 100:.2f} percentage points")

if __name__ == "__main__":
    run()

Running mathematical unit tests...
✓ AURC calculation passed.
✓ HCHR selective logic passed.
Unit tests passed.


--- LOADING DATASETS ---
Loaded 400 valid MCQs from afrimedqa_mcq_400.jsonl (Skipped 0 open-ended questions).
Loaded 400 valid MCQs from medqa_us_400.jsonl (Skipped 0 open-ended questions).

--- STARTING EVALUATION ---
Evaluating AfriMed-QA (n=400)...
[AfriMed-QA] Acc: 45.50% | AURC: 0.3403 | HCHR: 15.00%

Evaluating MedQA (n=400)...
[MedQA] Acc: 52.75% | AURC: 0.2904 | HCHR: 12.50%

--- FINAL SHIFT METRICS ---
Risk Inflation: 0.0499 (Lower AURC = safer selective behavior)
Hallucination Amplification: 2.50 percentage points
